# IDEA-002: MVRV Z-Score Entry Signal

**Hypothesis:** When MVRV Z-Score < 0, market is undervalued = buy.

**Logic:**
- MVRV Z-Score = (Market Cap - Realized Cap) / std(Market Cap)
- Z < 0 means market trading BELOW aggregate cost basis
- Historically rare - marks major cycle bottoms
- Different from SOPR: valuation metric vs sentiment metric

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("MVRV Z-Score Exploration 🔍")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

mvrv_z = pd.read_parquet(DATA_DIR / "mvrv_z.parquet").rename(columns={"value": "mvrv_z"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")

df = mvrv_z.join(mvrv, how='inner').join(price, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner')
df = df.sort_index()

print(f"Data: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

---
## 1. Understand the Data

In [ ]:
# Basic stats
print("MVRV Z-SCORE STATISTICS")
print("="*50)
print(f"Min: {df['mvrv_z'].min():.2f}")
print(f"Max: {df['mvrv_z'].max():.2f}")
print(f"Mean: {df['mvrv_z'].mean():.2f}")
print(f"Median: {df['mvrv_z'].median():.2f}")
print(f"Current: {df['mvrv_z'].iloc[-1]:.2f}")

print(f"\nPercentiles:")
for p in [5, 10, 25, 50, 75, 90, 95]:
    print(f"  {p}th: {df['mvrv_z'].quantile(p/100):.2f}")

In [ ]:
# How often is MVRV Z below various thresholds?
print("\nFREQUENCY BELOW THRESHOLDS")
print("="*50)
for thresh in [-0.5, 0, 0.5, 1.0, 1.5, 2.0]:
    days_below = (df['mvrv_z'] < thresh).sum()
    pct = days_below / len(df) * 100
    print(f"MVRV Z < {thresh}: {days_below} days ({pct:.1f}%)")

In [ ]:
# Visualize MVRV Z with price
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4],
                    subplot_titles=['BTC Price', 'MVRV Z-Score'])

fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['mvrv_z'], name='MVRV Z',
                         line=dict(color='purple')), row=2, col=1)

# Add threshold lines
fig.add_hline(y=0, line_dash='dash', line_color='red', row=2, col=1)
fig.add_hline(y=1, line_dash='dot', line_color='orange', row=2, col=1)
fig.add_hline(y=2, line_dash='dot', line_color='yellow', row=2, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=600, title_text='MVRV Z-Score - Buy Zone When < 0')
fig.show()

In [ ]:
# When was MVRV Z < 0?
low_z = df[df['mvrv_z'] < 0].copy()
print(f"\nPERIODS WITH MVRV Z < 0")
print("="*60)
print(f"Total days: {len(low_z)}")

if len(low_z) > 0:
    # Find distinct periods
    low_z['gap'] = (low_z.index.to_series().diff() > pd.Timedelta(days=30)).cumsum()
    periods = low_z.groupby('gap').agg({
        'mvrv_z': ['min', 'mean'],
        'price': ['first', 'last', 'min']
    })
    periods.columns = ['min_z', 'avg_z', 'start_price', 'end_price', 'min_price']

    period_dates = low_z.groupby('gap').apply(lambda x: (x.index.min(), x.index.max()))

    print(f"\nDistinct periods: {len(periods)}")
    print("\n" + "-"*80)
    for i, (idx, row) in enumerate(periods.iterrows()):
        start, end = period_dates.iloc[i]
        duration = (end - start).days + 1
        print(f"Period {i+1}: {start.date()} to {end.date()} ({duration} days)")
        print(f"  Z-Score: min={row['min_z']:.2f}, avg={row['avg_z']:.2f}")
        print(f"  Price: ${row['start_price']:,.0f} → ${row['end_price']:,.0f} (min: ${row['min_price']:,.0f})")
        print()
else:
    print("\n⚠️ MVRV Z never dropped below 0 in this dataset!")

---
## 2. Compare to SOPR Signal

In [ ]:
# How correlated are MVRV Z and SOPR?
print("CORRELATION ANALYSIS")
print("="*50)
print(f"MVRV Z vs SOPR: {df['mvrv_z'].corr(df['sopr']):.3f}")
print(f"MVRV Z vs STH SOPR: {df['mvrv_z'].corr(df['sopr_sth']):.3f}")
print(f"MVRV Z vs MVRV: {df['mvrv_z'].corr(df['mvrv']):.3f}")

# When SOPR signals fire, what's the MVRV Z?
sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
print(f"\nWhen SOPR double cap fires:")
print(f"  Avg MVRV Z: {df.loc[sopr_signal, 'mvrv_z'].mean():.2f}")
print(f"  Min MVRV Z: {df.loc[sopr_signal, 'mvrv_z'].min():.2f}")
print(f"  Max MVRV Z: {df.loc[sopr_signal, 'mvrv_z'].max():.2f}")

# When MVRV Z < 0, does SOPR also signal?
z_below_0 = df['mvrv_z'] < 0
if z_below_0.sum() > 0:
    sopr_overlap = (z_below_0 & sopr_signal).sum() / z_below_0.sum() * 100
    print(f"\nWhen MVRV Z < 0:")
    print(f"  SOPR also signaling: {sopr_overlap:.1f}% of the time")

In [ ]:
# Visualize both signals
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.4, 0.3, 0.3],
                    subplot_titles=['BTC Price', 'MVRV Z-Score', 'SOPR & STH SOPR'])

fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['mvrv_z'], name='MVRV Z'), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='red', row=2, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['sopr'], name='SOPR'), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['sopr_sth'], name='STH SOPR'), row=3, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=3, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=700, title_text='MVRV Z vs SOPR Signals')
fig.show()

---
## 3. Test MVRV Z as Entry Signal

In [ ]:
# Use same backtest framework
df_test = df[df.index >= '2018-12-15'].copy()
close = df_test['price']

def create_entry_signal(df, threshold):
    """Entry when MVRV Z drops below threshold (first day)."""
    below = df['mvrv_z'] < threshold
    entries = below & ~below.shift(1).fillna(False)
    return entries

In [ ]:
def backtest_mvrv_trailing(
    df, entries,
    mvrv_trigger=2.25,
    trailing_pct=0.20,
    stop_loss=0.20,
    max_hold_days=365
):
    """Same exit strategy as our best SOPR strategy."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            if not trailing_active and current_mvrv >= mvrv_trigger:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason,
            'entry_mvrv_z': df.loc[entry_date, 'mvrv_z']
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Test different MVRV Z thresholds
thresholds = [-0.5, -0.25, 0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]

print("MVRV Z THRESHOLD COMPARISON (In-Sample)")
print("="*100)
print(f"{'Threshold':<12} {'Signals':>10} {'Trades':>10} {'Return':>12} {'Win Rate':>10} {'Avg Days':>10}")
print("-"*100)

z_results = []

for thresh in thresholds:
    entries = create_entry_signal(df_test, thresh)
    n_signals = entries.sum()
    
    if n_signals == 0:
        print(f"Z < {thresh:<7} {n_signals:>10} {'-':>10} {'-':>12} {'-':>10} {'-':>10}")
        continue
    
    trades = backtest_mvrv_trailing(df_test, entries)
    
    total_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
    win_rate = (trades['pnl_pct'] > 0).mean() if len(trades) > 0 else 0
    avg_days = trades['days_held'].mean() if len(trades) > 0 else 0
    
    print(f"Z < {thresh:<7} {n_signals:>10} {len(trades):>10} {total_return*100:>11.0f}% "
          f"{win_rate*100:>9.0f}% {avg_days:>10.0f}")
    
    z_results.append({
        'threshold': thresh,
        'signals': n_signals,
        'trades': len(trades),
        'total_return': total_return,
        'win_rate': win_rate,
        'trades_df': trades
    })

In [ ]:
# Show trades for best threshold
if len(z_results) > 0:
    best_is = max(z_results, key=lambda x: x['total_return'])
    print(f"\nBest in-sample: Z < {best_is['threshold']}")
    print(f"Total return: {best_is['total_return']*100:.0f}%")
    print(f"\nTrades:")
    display_trades = best_is['trades_df'].copy()
    display_trades['entry_date'] = pd.to_datetime(display_trades['entry_date']).dt.strftime('%Y-%m-%d')
    display_trades['exit_date'] = pd.to_datetime(display_trades['exit_date']).dt.strftime('%Y-%m-%d')
    display_trades['pnl_pct'] = (display_trades['pnl_pct'] * 100).round(1)
    display_trades['entry_mvrv_z'] = display_trades['entry_mvrv_z'].round(2)
    print(display_trades[['entry_date', 'exit_date', 'entry_price', 'exit_price', 'pnl_pct', 'exit_reason', 'entry_mvrv_z']].to_string(index=False))

---
## 4. Walk-Forward Validation

In [ ]:
def walk_forward(df, threshold, mvrv_trigger=2.25, trailing_pct=0.20):
    """Walk-forward validation."""
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        entries = create_entry_signal(test_df, threshold)
        trades = backtest_mvrv_trailing(test_df, entries, mvrv_trigger, trailing_pct)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return,
            'n_trades': len(trades)
        })
    
    wf_df = pd.DataFrame(results)
    return wf_df['beat_hold'].mean(), (wf_df['strat_return'] - wf_df['hold_return']).mean(), wf_df['n_trades'].sum()

In [ ]:
# Walk-forward for each threshold
print("\nWALK-FORWARD VALIDATION")
print("="*80)
print(f"{'Threshold':<12} {'Beat Rate':>15} {'Avg Excess':>15} {'Total Trades':>15}")
print("-"*80)

wf_results = []

for thresh in thresholds:
    beat_rate, avg_excess, total_trades = walk_forward(df_test, thresh)
    
    print(f"Z < {thresh:<7} {beat_rate*100:>14.0f}% {avg_excess*100:>+14.1f}% {total_trades:>15}")
    
    wf_results.append({
        'threshold': thresh,
        'beat_rate': beat_rate,
        'avg_excess': avg_excess,
        'total_trades': total_trades
    })

wf_df = pd.DataFrame(wf_results)

In [ ]:
# Compare to SOPR baseline
print("\n\nCOMPARISON TO SOPR BASELINE")
print("="*60)
print(f"SOPR Double Cap + MVRV Trail: 62% beat rate")

valid_wf = wf_df[wf_df['total_trades'] > 0]
if len(valid_wf) > 0:
    best_idx = valid_wf['beat_rate'].idxmax()
    print(f"\nBest MVRV Z threshold: Z < {wf_df.loc[best_idx, 'threshold']}")
    print(f"Best MVRV Z beat rate: {wf_df.loc[best_idx, 'beat_rate']*100:.0f}%")
else:
    print("\n⚠️ No MVRV Z thresholds produced enough signals")

In [ ]:
# Visualize
valid_wf = wf_df[wf_df['total_trades'] > 0]

if len(valid_wf) > 0:
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=[f"Z < {t}" for t in valid_wf['threshold']],
        y=valid_wf['beat_rate'] * 100,
        marker_color=['green' if x > 0.62 else 'orange' if x > 0.54 else 'gray' for x in valid_wf['beat_rate']],
        text=[f"{x:.0f}%" for x in valid_wf['beat_rate']*100],
        textposition='outside'
    ))

    fig.add_hline(y=54, line_dash='dash', line_color='orange', annotation_text='SOPR baseline 54%')
    fig.add_hline(y=62, line_dash='dash', line_color='green', annotation_text='SOPR+MVRV 62%')

    fig.update_layout(
        title='Walk-Forward Beat Rate by MVRV Z Threshold',
        yaxis_title='Beat Rate %',
        height=500
    )
    fig.show()
else:
    print("Not enough data to visualize")

---
## 5. Combine MVRV Z with SOPR?

In [ ]:
# What if we require BOTH signals?
# SOPR < 1 AND STH SOPR < 1 AND MVRV Z < threshold

def combined_entry(df, z_threshold):
    """Entry when SOPR double cap AND MVRV Z below threshold."""
    sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
    z_signal = df['mvrv_z'] < z_threshold
    combined = sopr_signal & z_signal
    entries = combined & ~combined.shift(1).fillna(False)
    return entries

print("COMBINED SIGNAL: SOPR + MVRV Z")
print("="*80)
print(f"{'Z Threshold':<15} {'Signals':>10} {'Beat Rate':>15} {'Avg Excess':>15}")
print("-"*80)

combined_results = []

for z_thresh in [-0.5, 0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    # Walk-forward
    results = []
    close = df_test['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df_test)
    n_folds = (total_days - train_days) // step_days
    total_signals = 0
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df_test.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        entries = combined_entry(test_df, z_thresh)
        total_signals += entries.sum()
        trades = backtest_mvrv_trailing(test_df, entries)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    wf_result = pd.DataFrame(results)
    beat_rate = wf_result['beat_hold'].mean()
    avg_excess = (wf_result['strat_return'] - wf_result['hold_return']).mean()
    
    print(f"SOPR + Z<{z_thresh:<5} {total_signals:>10} {beat_rate*100:>14.0f}% {avg_excess*100:>+14.1f}%")
    
    combined_results.append({
        'z_threshold': z_thresh,
        'signals': total_signals,
        'beat_rate': beat_rate,
        'avg_excess': avg_excess
    })

---
## 6. Alternative: Use MVRV Z as Replacement for MVRV Exit

In [ ]:
# What if we use MVRV Z for EXIT instead of entry?
# SOPR entry + MVRV Z > threshold triggers trailing stop

def backtest_mvrv_z_exit(
    df, entries,
    z_trigger=3.0,  # Z-score that triggers trailing stop
    trailing_pct=0.20,
    stop_loss=0.20,
    max_hold_days=365
):
    """Use MVRV Z-Score to trigger exit."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_z = df['mvrv_z'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            # Use MVRV Z instead of regular MVRV
            if not trailing_active and current_z >= z_trigger:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'z_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'pnl_pct': pnl,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

# Test MVRV Z as exit trigger
print("\n\nMVRV Z AS EXIT TRIGGER (with SOPR entry)")
print("="*80)
print(f"{'Z Exit Trigger':<15} {'Beat Rate':>15} {'Avg Excess':>15}")
print("-"*80)

# Create SOPR entry signal
sopr_entries = (df_test['sopr'] < 1) & (df_test['sopr_sth'] < 1)
sopr_entries = sopr_entries & ~sopr_entries.shift(1).fillna(False)

z_exit_results = []

for z_trigger in [2.0, 2.5, 3.0, 3.5, 4.0, 5.0]:
    # Walk-forward
    results = []
    close = df_test['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df_test)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df_test.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        test_entries = sopr_entries.iloc[test_start:test_end]
        
        trades = backtest_mvrv_z_exit(test_df, test_entries, z_trigger=z_trigger)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    wf_result = pd.DataFrame(results)
    beat_rate = wf_result['beat_hold'].mean()
    avg_excess = (wf_result['strat_return'] - wf_result['hold_return']).mean()
    
    print(f"Z > {z_trigger:<10} {beat_rate*100:>14.0f}% {avg_excess*100:>+14.1f}%")
    
    z_exit_results.append({
        'z_trigger': z_trigger,
        'beat_rate': beat_rate,
        'avg_excess': avg_excess
    })

---
## 7. Summary

In [ ]:
print("\n" + "="*80)
print("MVRV Z-SCORE ANALYSIS SUMMARY")
print("="*80)

# Best standalone MVRV Z entry
valid_wf = wf_df[wf_df['total_trades'] > 0]
if len(valid_wf) > 0:
    best_z = valid_wf.loc[valid_wf['beat_rate'].idxmax()]
    print(f"\n📊 STANDALONE MVRV Z ENTRY")
    print(f"   Best threshold: Z < {best_z['threshold']}")
    print(f"   Beat rate: {best_z['beat_rate']*100:.0f}%")
    print(f"   Avg excess: {best_z['avg_excess']*100:+.1f}%")
else:
    print(f"\n📊 STANDALONE MVRV Z ENTRY")
    print(f"   ⚠️ Not enough signals")
    best_z = None

# Best combined
combined_df = pd.DataFrame(combined_results)
valid_combined = combined_df[combined_df['signals'] > 0]
if len(valid_combined) > 0:
    best_combined = valid_combined.loc[valid_combined['beat_rate'].idxmax()]
    print(f"\n📊 COMBINED SIGNAL (SOPR + MVRV Z)")
    print(f"   Best threshold: SOPR + Z < {best_combined['z_threshold']}")
    print(f"   Beat rate: {best_combined['beat_rate']*100:.0f}%")
    print(f"   Avg excess: {best_combined['avg_excess']*100:+.1f}%")
else:
    print(f"\n📊 COMBINED SIGNAL (SOPR + MVRV Z)")
    print(f"   ⚠️ Not enough signals")
    best_combined = None

# Best Z exit
z_exit_df = pd.DataFrame(z_exit_results)
best_z_exit = z_exit_df.loc[z_exit_df['beat_rate'].idxmax()]
print(f"\n📊 MVRV Z AS EXIT TRIGGER")
print(f"   Best threshold: Z > {best_z_exit['z_trigger']}")
print(f"   Beat rate: {best_z_exit['beat_rate']*100:.0f}%")
print(f"   Avg excess: {best_z_exit['avg_excess']*100:+.1f}%")

# Comparison
print(f"\n📊 COMPARISON")
print(f"   SOPR + MVRV exit:  62% beat rate (current best)")
if best_z is not None:
    print(f"   MVRV Z entry:      {best_z['beat_rate']*100:.0f}% beat rate")
if best_combined is not None:
    print(f"   SOPR + Z filter:   {best_combined['beat_rate']*100:.0f}% beat rate")
print(f"   SOPR + Z exit:     {best_z_exit['beat_rate']*100:.0f}% beat rate")

# Verdict
print(f"\n🎯 VERDICT:")
improvement = False
if best_z is not None and best_z['beat_rate'] > 0.62:
    print(f"   ✅ MVRV Z entry BEATS SOPR!")
    improvement = True
if best_combined is not None and best_combined['beat_rate'] > 0.62:
    print(f"   ✅ Combined signal BEATS baseline!")
    improvement = True
if best_z_exit['beat_rate'] > 0.62:
    print(f"   ✅ MVRV Z exit BEATS regular MVRV exit!")
    improvement = True
if not improvement:
    print(f"   ⚠️ MVRV Z doesn't improve on current best. Stick with SOPR + MVRV > 2.25 trail.")

print("\n" + "="*80)

In [ ]:
# Save results
import json

def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_native(v) for v in obj]
    elif hasattr(obj, 'item'):
        return obj.item()
    return obj

results = {
    'signal': 'mvrv_z_score',
    'standalone_entry_results': to_native(wf_df.to_dict('records')),
    'combined_results': to_native(combined_results),
    'z_exit_results': to_native(z_exit_results),
    'best_standalone': to_native(dict(best_z)) if best_z is not None else None,
    'best_combined': to_native(dict(best_combined)) if best_combined is not None else None,
    'best_z_exit': to_native(dict(best_z_exit)),
    'baseline_sopr': {'beat_rate': 0.62}
}

with open('../data/mvrv_z_score_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to ../data/mvrv_z_score_results.json")